# Workshop 3 · 01 — Stammdaten heilen (Aufgabe)

**Ziel:** Unvollständige und uneinheitliche Stammdaten sowie Freitextmeldungen in **saubere, strukturierte Spalten** verwandeln — deklarativ mit **AI Functions**, ohne eigenes Modelltraining.

**Ausgangslage:** Tabelle `asset_meldungen` (aus Notebook 00) mit uneinheitlichen `baureihe`/`depot`, fehlender `komponente` und einem `freitext_meldung`, in dem die eigentliche Information steckt.

**Deine Aufgaben:**
1. **Normalisieren** — `depot` und `baureihe` mit `ai.classify` auf einheitliche Werte mappen.
2. **Extrahieren** — fehlende Felder (`komponente`, `ort`, `schweregrad`, `sicherheitsrelevant`, `prioritaet`) mit `ai.generate_response` + JSON-Schema aus dem Freitext ziehen.
3. Ergebnis als Tabelle `asset_clean` speichern.

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/overview

## Schritt 0 — Daten ansehen
Führe die Zelle aus, um die uneinheitlichen Rohdaten zu sehen: teils leere/uneinheitliche Stammdaten, die Information steckt nur im Freitext.

In [ ]:
import synapse.ml.spark.aifunc as aifunc  # registriert den .ai Accessor auf Spark-DataFrames

df = spark.table('asset_meldungen')
display(df.select('meldung_id', 'baureihe', 'depot', 'komponente', 'freitext_meldung'))

## Schritt 1 — Stammdaten normalisieren mit `ai.classify`

`depot` (frei geschrieben) und `baureihe` (Tippfehler, Schreibvarianten, teils leer) sollen auf **einheitliche Werte** gemappt werden — eine Zeile Code statt 50 `CASE WHEN`.

**Aufgabe:** Erzeuge die Spalten `depot_norm` und `baureihe_norm`.

*Hinweis:* `df.ai.classify(input_col=..., output_col=..., labels=[...])` — die Aufrufe lassen sich verketten. Überlege dir sinnvolle Ziel-Labels.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO 1: Definiere Label-Listen (einheitliche Zielwerte) fuer Depots und Baureihen.
depot_labels = ...
baureihe_labels = ...

# TODO 2: Erzeuge mit df.ai.classify(...) die Spalten 'depot_norm' und 'baureihe_norm'.
norm = ...

# TODO 3: Zeige zur Kontrolle meldung_id, depot, depot_norm, baureihe, baureihe_norm an.

## Schritt 2 — fehlende Felder aus dem Freitext ziehen

`ai.generate_response` mit einem **JSON-Schema** liefert pro Zeile **typisierte Felder** statt Freitext-Raten.

**Aufgabe:**
1. Definiere ein JSON-Schema mit den Feldern `komponente` (string), `ort_am_fahrzeug` (string), `schweregrad` (integer 1–5), `sicherheitsrelevant` (boolean), `prioritaet` (enum: hoch/mittel/niedrig).
2. Rufe `norm.ai.generate_response(...)` mit `is_prompt_template=True`, deinem `response_format` und `output_col='extract_json'` auf. Der Prompt referenziert `{freitext_meldung}`.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO 1: JSON-Schema fuer die zu extrahierenden Felder definieren.
meldung_schema = ...

# TODO 2: norm.ai.generate_response(...) mit Prompt-Template auf {freitext_meldung} aufrufen
#         (is_prompt_template=True, response_format=meldung_schema, output_col='extract_json').
extracted = ...

# TODO 3: Zeige meldung_id, freitext_meldung, extract_json an.

## Schritt 3 — JSON parsen und als Tabelle speichern

**Aufgabe:** Parse die JSON-Spalte `extract_json` in typisierte Spalten und speichere das Endergebnis als Tabelle `asset_clean` (Spalten: `meldung_id`, `wagon_id`, `baureihe_norm`, `depot_norm`, `komponente`, `ort`, `schweregrad`, `sicherheitsrelevant`, `prioritaet`, `freitext_meldung`, `kunden_kommentar`, `foto_pfad`).

*Hinweis:* `from_json(col('extract_json'), <StructType>)` liefert eine Struct-Spalte, aus der du die Felder selektierst.

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

# TODO 1: Struktur-Schema (StructType) passend zum JSON definieren.
# TODO 2: extract_json parsen und gewuenschte Spalten selektieren -> clean
# TODO 3: clean als Tabelle 'asset_clean' speichern (mode overwrite, overwriteSchema true).

### Ergebnis-Check
Wenn alles klappt, hast du die Tabelle `asset_clean` mit normalisierten Stammdaten und aus Freitext extrahierten Feldern. Prüfe: höchster `schweregrad` oben, keine leeren `depot_norm`/`komponente`.

> **Kurzform** für reine Entitäts-Extraktion: `df.ai.extract(input_col='freitext_meldung', labels=['komponente', 'ort', 'schweregrad'])` liefert je Label eine Spalte.